In [3]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('motorola.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai plus', 'aiplus'],
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [4]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"MOTOROLA g37 power (PANTONE Nautical Blue, 128...",15999.0,30499.0,4.2,842 Ratings & 141 Reviews,4,128,16.92 cm (6.66 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,7000,Dimensity 6400 Processor,https://www.flipkart.com/motorola-g37-power-pa...,https://rukminim2.flixcart.com/image/312/312/x...,47,motorola,842.0,141.0
1,MOTOROLA Edge 60 Fusion 5G (PANTONE Slipstream...,22999.0,24999.0,4.4,"1,23,808 Ratings & 7,255 Reviews",8,128,16.94 cm (6.67 inch) Display,50MP + 13MP | 32MP Front Camera,5500,Dimensity 7400 Processor,https://www.flipkart.com/motorola-edge-60-fusi...,https://rukminim2.flixcart.com/image/312/312/x...,8,motorola,123808.0,7255.0
2,"MOTOROLA g37 (PANTONE Nautical Blue, 64 GB)",14999.0,24999.0,4.3,756 Ratings & 47 Reviews,4,64,16.92 cm (6.66 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,5200,Dimensity 6400 Processor,https://www.flipkart.com/motorola-g37-pantone-...,https://rukminim2.flixcart.com/image/312/312/x...,40,motorola,756.0,47.0
3,MOTOROLA g57 power 5G Snapdragon (Pantone Rega...,20999.0,21999.0,4.4,"48,242 Ratings & 2,883 Reviews",8,128,17.07 cm (6.72 inch) Full HD+ Display,50MP + 8MP | 8MP Front Camera,7000,6s Gen 4 Processor,https://www.flipkart.com/motorola-g57-power-5g...,https://rukminim2.flixcart.com/image/312/312/x...,4,motorola,48242.0,2883.0
4,"MOTOROLA g37 (PANTONE Impenetrable, 64 GB)",14999.0,24999.0,4.3,756 Ratings & 47 Reviews,4,64,16.92 cm (6.66 inch) HD+ Display,50MP Rear Camera | 8MP Front Camera,5200,Dimensity 6400 Processor,https://www.flipkart.com/motorola-g37-pantone-...,https://rukminim2.flixcart.com/image/312/312/x...,40,motorola,756.0,47.0


In [5]:
df.isnull().sum()

Product Name         0
Current Price        1
MRP                103
Rating               3
Review text          3
RAM                  0
Storage              0
Display              0
Camera               3
Battery              0
Processor           14
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              3
Reviews              3
dtype: int64

In [6]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [7]:
df.isnull().sum()

Product Name        0
Current Price       1
MRP                 1
Rating              3
Review text         3
RAM                 0
Storage             0
Display             0
Camera              3
Battery             0
Processor          14
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             3
Reviews             3
dtype: int64

In [8]:
def get_motorola_camera(product_name):
    if pd.isna(product_name):
        return 'Unknown Camera'
    
    # Edge 50 Series
    if 'Edge 50 Ultra' in product_name:
        return '50MP + 50MP + 64MP'
    elif 'Edge 50 Pro' in product_name:
        return '50MP + 13MP + 10MP'
    elif 'Edge 50 Fusion' in product_name:
        return '50MP + 13MP'
    elif 'Edge 50' in product_name:
        return '50MP + 13MP'
    
    # Edge 40 Series
    elif 'Edge 40' in product_name:
        return '50MP + 13MP'
    
    # Edge 30 Series
    elif 'Edge 30' in product_name:
        if 'Ultra' in product_name:
            return '200MP + 50MP + 12MP'
        elif 'Pro' in product_name:
            return '50MP + 50MP'
        else:
            return '50MP + 13MP'
    
    # G Series
    elif 'G85' in product_name:
        return '50MP + 8MP + 2MP'
    elif 'G84' in product_name:
        return '50MP + 8MP'
    elif 'G73' in product_name:
        return '50MP + 8MP'
    elif 'G64' in product_name:
        return '50MP + 8MP'
    elif 'G54' in product_name:
        return '50MP + 8MP'
    elif 'G53' in product_name:
        return '50MP + 2MP'
    elif 'G45' in product_name:
        return '50MP + 2MP'
    elif 'G34' in product_name:
        return '50MP + 2MP'
    elif 'G24' in product_name:
        return '50MP + 2MP'
    elif 'G14' in product_name:
        return '50MP + 2MP'
    elif 'G' in product_name:
        return '50MP + 2MP'
    
    # Razr Series
    elif 'Razr' in product_name or 'RAZR' in product_name:
        return '12MP + 12MP'
    
    # E Series
    elif 'E' in product_name:
        return '13MP + 2MP'
    
    else:
        return 'Unknown Camera'

df.loc[df['Camera'].isnull(), 'Camera'] = df.loc[df['Camera'].isnull(), 'Product Name'].apply(get_motorola_camera)

In [9]:
df.isnull().sum()

Product Name        0
Current Price       1
MRP                 1
Rating              3
Review text         3
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor          14
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             3
Reviews             3
dtype: int64

In [10]:
def get_motorola_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # Edge 50 Series
    if 'Edge 50 Ultra' in product_name:
        return 'Snapdragon 8s Gen 3'
    elif 'Edge 50 Pro' in product_name:
        return 'Snapdragon 7 Gen 3'
    elif 'Edge 50 Fusion' in product_name:
        return 'Snapdragon 7s Gen 2'
    elif 'Edge 50' in product_name:
        return 'Snapdragon 7 Gen 1'
    
    # Edge 40 Series
    elif 'Edge 40' in product_name:
        return 'MediaTek Dimensity 7030'
    
    # Edge 30 Series
    elif 'Edge 30 Ultra' in product_name:
        return 'Snapdragon 8+ Gen 1'
    elif 'Edge 30 Pro' in product_name:
        return 'Snapdragon 8 Gen 1'
    elif 'Edge 30' in product_name:
        return 'Snapdragon 778G+'
    
    # G Series
    elif 'G85' in product_name:
        return 'Snapdragon 6s Gen 3'
    elif 'G84' in product_name:
        return 'Snapdragon 695'
    elif 'G73' in product_name:
        return 'MediaTek Dimensity 930'
    elif 'G64' in product_name:
        return 'MediaTek Dimensity 7020'
    elif 'G54' in product_name:
        return 'MediaTek Dimensity 7020'
    elif 'G53' in product_name:
        return 'Snapdragon 480+'
    elif 'G45' in product_name:
        return 'Snapdragon 6s Gen 3'
    elif 'G34' in product_name:
        return 'Snapdragon 695'
    elif 'G24' in product_name:
        return 'MediaTek Helio G85'
    elif 'G14' in product_name:
        return 'MediaTek Helio G85'
    elif 'G' in product_name:
        return 'Snapdragon 6 Series'
    
    # Razr Series
    elif 'Razr 50' in product_name or 'RAZR 50' in product_name:
        return 'MediaTek Dimensity 7300X'
    elif 'Razr 40' in product_name or 'RAZR 40' in product_name:
        return 'Snapdragon 7 Gen 1'
    
    # E Series
    elif 'E' in product_name:
        return 'MediaTek Helio G37'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_motorola_processor)

In [11]:
df.isnull().sum()

Product Name       0
Current Price      1
MRP                1
Rating             3
Review text        3
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            3
Reviews            3
dtype: int64

In [12]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

In [13]:
df.isnull().sum()

Product Name       0
Current Price      1
MRP                1
Rating             0
Review text        3
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [15]:
null_product = df[df['Current Price'].isnull()]['Product Name'].iloc[0]
print(f"Product with null price: {null_product}")

Product with null price: Motorola e13 (Creamy White, 64 GB)


In [16]:
# Fill the specific product with correct price
df.loc[df['Product Name'] == 'Motorola e13 (Creamy White, 64 GB)', 'Current Price'] = 7500  # Approximate price
df.loc[df['Product Name'] == 'Motorola e13 (Creamy White, 64 GB)', 'MRP'] = 9999  # Approximate MRP

In [17]:
df.isnull().sum()

Product Name         0
Current Price        0
MRP                  0
Rating               0
Review text          3
RAM                  0
Storage              0
Display              0
Camera               0
Battery              0
Processor            0
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              0
Reviews              0
Motorola_Model     245
dtype: int64

In [18]:
df = df.drop('Motorola_Model', axis=1)

In [19]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        3
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [20]:
df.to_excel('Cleaned_motorola.xlsx', index=False)